# 03 — Standalone Explainability & Faithfulness Reference
> [!NOTE]
> **Pipeline Consolidation Notice:**
> The complete end-to-end research workflow—from environment verification, dataset loading, and model training to multimodal Grad-CAM++, demographic SHAP, GNNExplainer, 3-way modality attribution, and quantitative faithfulness curves—is now completely consolidated into:
> 👉 **`notebooks/02_model_training.ipynb` (Cells 1 to 16)**.
>
> Notebook 03 is no longer required for executing the full research pipeline. It is preserved here as a standalone reference for independent post-hoc explainability evaluations on saved checkpoints.

This standalone reference notebook implements:
1. **Image Explainability (Grad-CAM++ & ViT)**: Disease-specific saliency heatmaps overlaid on original CXR scans for CNN and Vision Transformer backbones.
2. **Tabular Explainability (SHAP)**: Exact analytical Shapley feature attributions for clinical metadata (Age, Sex) across all 14 thoracic disease targets without target label leakage.
3. **Scan Graph Explainability (GNNExplainer)**: Longitudinal patient scan-graph edge and node importance identifying influential prior and concurrent scans.
4. **Disease-Specific Multimodal Fusion Attribution**: Exact 3-player Shapley attribution (Image vs Clinical Meta vs Scan Graph) quantifying modality reliance for each disease head.
5. **Quantitative Faithfulness Evaluation**: Pixel deletion/insertion curves, tabular feature perturbation tests, graph edge drop tests, and modality ablation benchmarks.

> **Important Clinical Disclaimer:** Modality and feature attributions reflect mathematical representation sensitivity within the trained network and do not establish clinical causality.


## Cell 1 — Environment Setup & Dependencies

In [ ]:
import subprocess, sys
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

pkgs = [
    'timm',
    'datasets',
    'scikit-learn',
    'pandas',
    'matplotlib',
    'PyYAML',
    'Pillow',
    'pennylane',
    'pennylane-lightning',
    'torch-geometric',
    'networkx',
    'scipy',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All required dependencies are installed.')

## Cell 2 — Mount Google Drive / Clone Repository

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Option A: Google Colab Environment ────────────────────────────────────────
if os.path.exists('/content'):
    REPO_URL = 'https://github.com/pchandraswaroop/UROP.git'
    REPO_PATH = '/content/UROP'
    TARGET_BRANCH = os.environ.get('UROP_BRANCH', 'execution-readiness')
    
    if not os.path.exists(REPO_PATH):
        print(f'Cloning repository {REPO_URL} (branch: {TARGET_BRANCH}) into {REPO_PATH}...')
        subprocess.check_call(['git', 'clone', '--branch', TARGET_BRANCH, REPO_URL, REPO_PATH])
    os.chdir(REPO_PATH)
    print(f'Current Working Directory: {os.getcwd()}')
else:
    # Option B: Local or Drive execution
    REPO_PATH = os.getcwd()
    print(f'Working in repository directory: {REPO_PATH}')

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)


## Cell 3 — Load Model & Test Records

In [ ]:
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import torch

from src.models.model_factory import build_model
from src.dataset.label_encoder import DISEASE_LABEL_NAMES, safe_parse_labels, NIH_LABEL_NAMES
from src.dataset.preprocessor import MetadataEncoder
from src.graph.graph_builder import build_batch_scan_graph
from src.training.trainer import CheckpointManager

# ── 1. Configuration & Mode Selection ─────────────────────────────────────────
RUN_MODE = 'smoke'  # 'smoke', 'medium', 'full', or 'full_50'
ALLOW_UNWEIGHTED_DEBUG = False

with open('config/config.yaml') as f:
    cfg = yaml.safe_load(f)

SPLIT_DIR = cfg['splits']['split_dir']   # 'data/splits'
HF_REPO   = cfg['dataset']['hf_repo']   # 'chehablab/NIHChestXR'
HF_CACHE  = cfg['dataset']['cache_dir'] # 'data/raw'

print('Disease Targets (14):', DISEASE_LABEL_NAMES)

# ── 2. Build Model & Discover Checkpoint ──────────────────────────────────────
model_name = 'quantum_gnn_multimodal'
model = build_model(model_name, pretrained=False, device=device)

ckpt_candidates_dirs = [
    Path(f'checkpoints/{RUN_MODE}/{model_name}'),
    Path(f'checkpoints/{model_name}'),
]

best_checkpoint_path = None
best_val_auc = -1.0
best_epoch = -1

for cd in ckpt_candidates_dirs:
    if cd.exists():
        found = list(cd.glob('checkpoint_epoch*.pt'))
        if found:
            ranked = []
            for p in found:
                try:
                    payload = torch.load(p, map_location='cpu', weights_only=False)
                    v_score = payload.get('metrics', {}).get('val_auc_mean', -1.0)
                    ep = payload.get('epoch', -1)
                    ranked.append((v_score, ep, p))
                except Exception:
                    pass
            if ranked:
                ranked.sort(key=lambda x: x[0], reverse=True)
                best_val_auc, best_epoch, best_checkpoint_path = ranked[0]
                break

if best_checkpoint_path is not None:
    CheckpointManager.load(str(best_checkpoint_path), model, device=device)
    print(f'Loaded verified checkpoint for {model_name} [{RUN_MODE.upper()}]:')
    print(f' - Path:         {best_checkpoint_path.resolve()}')
    print(f' - Best Val AUC: {best_val_auc:.4f} (Epoch {best_epoch + 1 if isinstance(best_epoch, int) else best_epoch})')
else:
    if not ALLOW_UNWEIGHTED_DEBUG:
        raise FileNotFoundError(
            f"No trained checkpoint found in 'checkpoints/{RUN_MODE}/{model_name}'!\n"
            f"Explainability saliency maps and SHAP attributions require a trained model checkpoint.\n"
            f"Please complete training in Notebook 02 first, or set ALLOW_UNWEIGHTED_DEBUG = True "
            f"if you only want to test code pipeline execution."
        )
    print('⚠ WARNING: No trained checkpoint found. Running with randomly initialized weights (DEBUG ONLY).')

model.eval()
print(f'Ready: {model_name} on {device}')

# ── 3. Load Held-Out Test Records (Strict 0% Leakage) ─────────────────────────
test_csv_path = Path(SPLIT_DIR) / 'test.csv'
assert test_csv_path.exists(), f'Held-out test split not found at {test_csv_path}!'
test_df = pd.read_csv(test_csv_path)

# Verify no overlap with train or val splits
for s_name in ['train.csv', 'val.csv']:
    s_path = Path(SPLIT_DIR) / s_name
    if s_path.exists():
        s_df = pd.read_csv(s_path)
        overlap = set(test_df['patient_id'].unique()).intersection(set(s_df['patient_id'].unique()))
        assert len(overlap) == 0, f'Leakage detected: {len(overlap)} patients overlap between test and {s_name}!'
print(f'Verified held-out test split: {len(test_df):,} scans across {test_df["patient_id"].nunique():,} unique patients (0% leakage).')

# ── 4. Build Test Evaluation Cohort (Real Patients with Companion Scans) ──────
p_counts = test_df['patient_id'].value_counts()
multi_pids = p_counts[p_counts >= 2].index
single_pids = p_counts[p_counts == 1].index

# Select a multi-scan test patient (2 companion scans)
selected_multi_pid = multi_pids[0]
df_multi = test_df[test_df['patient_id'] == selected_multi_pid].head(2)

# Select two single-scan test patients for cohort context
df_singles = test_df[test_df['patient_id'].isin(single_pids[:2])].drop_duplicates(subset=['patient_id'])

# Combine into held-out evaluation cohort
cohort_df = pd.concat([df_multi, df_singles]).reset_index(drop=True)
patient_ids = [str(pid) for pid in cohort_df['patient_id']]
scan_ids = [str(sid) for sid in cohort_df['scan_id']]

print(f'\nConstructed held-out evaluation cohort ({len(cohort_df)} scans, {cohort_df["patient_id"].nunique()} patients):')
for i, r in cohort_df.iterrows():
    raw_l = r['labels_raw'] if 'labels_raw' in r and pd.notna(r['labels_raw']) else r['labels']
    p_labels = safe_parse_labels(raw_l)
    label_str = ', '.join([NIH_LABEL_NAMES[idx] for idx in p_labels if 0 < idx < len(NIH_LABEL_NAMES)]) or 'No Finding'
    print(f' - Scan {i} [Patient {r["patient_id"]}, Scan {r["scan_id"]}]: Age {r["age"]}, Sex {r["sex"]} | Ground Truth: {label_str}')

# ── 5. Load Real Radiographs from HuggingFace Dataset ─────────────────────────
print(f'\nLoading radiographs for cohort from {HF_REPO}...')
try:
    from datasets import load_dataset
    hf_dataset = load_dataset(HF_REPO, split='train', cache_dir=HF_CACHE)
    raw_images_pil = [hf_dataset[int(r['hf_index'])]['image'].convert('RGB') for _, r in cohort_df.iterrows()]
    print(f'✓ Successfully loaded {len(raw_images_pil)} real patient chest radiographs.')
except Exception as e:
    print(f'Notice: HuggingFace dataset download skipped or offline ({e}).')
    if not ALLOW_UNWEIGHTED_DEBUG:
        raise RuntimeError(
            f'Unable to load genuine test radiographs from {HF_REPO}.\n'
            f'If executing offline without internet access, set ALLOW_UNWEIGHTED_DEBUG = True.'
        )
    print('⚠ Using synthetic placeholder images for code pipeline verification.')
    raw_images_pil = [Image.fromarray(np.uint8(np.random.rand(224, 224, 3) * 255)) for _ in range(len(cohort_df))]

# Preprocess images to standard ImageNet normalized tensors
val_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
sample_images = torch.stack([val_transforms(img) for img in raw_images_pil]).to(device)
raw_images_np = [np.array(img.resize((224, 224))) for img in raw_images_pil]

# Preprocess metadata using standardized MetadataEncoder: (clamp(age, 0, 120) - 46.9) / 16.6 | M=1.0, F=0.0
meta_encoder = MetadataEncoder()
sample_meta = meta_encoder.encode_batch(
    ages=[int(r['age']) for _, r in cohort_df.iterrows()],
    sexes=[str(r['sex']) for _, r in cohort_df.iterrows()]
).to(device)

# Build real patient-level scan graph with directed edges and self-loops
gx, edge_index = build_batch_scan_graph(sample_meta.cpu(), patient_ids, add_self_loops=True)
gx = gx.to(device)
edge_index = edge_index.to(device)
print(f'Real Patient Graph: {gx.shape[0]} nodes, {edge_index.shape[1]} directed edges across {len(set(patient_ids))} patients.')


## Cell 4 — Image Explainability (Grad-CAM++ & ViT Attribution)
Generates disease-specific spatial heatmaps and overlays for selected pathology targets (e.g. Cardiomegaly, Pneumothorax).

In [ ]:
import matplotlib.pyplot as plt
from src.explainability.gradcam import get_image_explainer
from src.explainability.visualizer import ExplainabilityVisualizer
from src.dataset.label_encoder import DISEASE_LABEL_NAMES

img_explainer = get_image_explainer(model)

# Primary scan visual saliency (Patient's primary radiograph)
raw_img = raw_images_np[0]

# Determine target diseases: inspect model prediction on primary scan
with torch.no_grad():
    primary_logits = model(
        sample_images[:1],
        sample_meta[:1],
        graph_x=sample_meta[:1],
        graph_edge_index=torch.zeros((2, 1), dtype=torch.long, device=device),
        return_logits=True
    )
    primary_probs = torch.sigmoid(primary_logits)[0].cpu().numpy()

# Select top-2 predicted pathologies to explain visually
top_disease_indices = np.argsort(primary_probs)[::-1][:2]
target_diseases = [(DISEASE_LABEL_NAMES[i], int(i)) for i in top_disease_indices]

print(f'Top model predictions for primary radiograph (Scan {scan_ids[0]}):')
for d_name, c_idx in target_diseases:
    print(f' - {d_name:<20}: {primary_probs[c_idx]*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_img)
axes[0].set_title(f'Primary Chest Radiograph\n[Patient {patient_ids[0]}, Scan {scan_ids[0]}]', fontsize=11, fontweight='bold')
axes[0].axis('off')

for idx, (disease_name, class_idx) in enumerate(target_diseases, start=1):
    hmap = img_explainer.generate(
        sample_images[:1],
        sample_meta[:1],
        target_class=class_idx,
        graph_x=sample_meta[:1],
        graph_edge_index=torch.zeros((2, 1), dtype=torch.long, device=device)
    )[0]
    overlay = ExplainabilityVisualizer.overlay_heatmap_on_image(raw_img, hmap, alpha=0.45)
    axes[idx].imshow(overlay)
    axes[idx].set_title(f'Grad-CAM++: {disease_name}\n(Prob: {primary_probs[class_idx]*100:.1f}%)', fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()


## Cell 5 — Tabular Explainability (Exact Analytical Shapley Values)
Quantifies the impact of clinical metadata (`Age` and `Sex`) on predictions without target label leakage.

In [ ]:
from src.explainability.tabular_shap import TabularFeatureExplainer
from src.explainability.visualizer import ExplainabilityVisualizer
from src.dataset.label_encoder import DISEASE_LABEL_NAMES

tab_explainer = TabularFeatureExplainer(model, num_classes=14)
# Pass primary scan with valid self-loop
shap_results = tab_explainer.explain_instance(
    sample_images[:1],
    sample_meta[:1],
    graph_x=sample_meta[:1],
    graph_edge_index=torch.zeros((2, 1), dtype=torch.long, device=device),
)

target_disease_name = target_diseases[0][0]
target_disease_idx = target_diseases[0][1]

print(f'Exact Analytical Demographic Shapley Attributions for Scan {scan_ids[0]} (Top 5 Diseases):')
for d_idx in range(5):
    d_name = DISEASE_LABEL_NAMES[d_idx]
    age_val = shap_results['Age'][0, d_idx]
    sex_val = shap_results['Sex'][0, d_idx]
    print(f'  {d_name:<20}: Age SHAP = {age_val:+.4f}, Sex SHAP = {sex_val:+.4f}')

fig = ExplainabilityVisualizer.plot_tabular_shap(
    shap_results,
    sample_idx=0,
    disease_name=target_disease_name,
    disease_idx=target_disease_idx,
    scan_id=scan_ids[0]
)
plt.show()


## Cell 6 — Scan Graph Explainability (GNNExplainer)
Visualizes scan-to-scan connectivity and isolates influential neighboring scans for multi-scan patients.

In [ ]:
from src.explainability.gnn_explainer import ScanGraphExplainer

graph_explainer = ScanGraphExplainer(model, num_classes=14)
target_disease_name = target_diseases[0][0]
target_disease_idx = target_diseases[0][1]

# Explain actual multi-scan cohort graph
graph_exp = graph_explainer.explain(
    sample_images,
    sample_meta,
    gx,
    edge_index,
    target_class=target_disease_idx,
    patient_ids=patient_ids,
    scan_ids=scan_ids
)

print(f"GNN Inter-Scan Edge Attributions for '{target_disease_name}':")
for u, v, w in zip(graph_exp['edge_index'][0], graph_exp['edge_index'][1], graph_exp['edge_importance']):
    rel_type = 'Self-Loop' if u == v else 'Inter-Scan Companion Edge'
    print(f'  Scan {scan_ids[u]} (Pt {patient_ids[u]}) -> Scan {scan_ids[v]} (Pt {patient_ids[v]}) [{rel_type}]: Importance = {w:.4f}')

fig = graph_explainer.visualize_graph(graph_exp, title=f'Patient Scan Graph Explanation ({target_disease_name})')
plt.show()


## Cell 7 — Multimodal Fusion Attribution (Image vs Clinical Meta vs Scan Graph)
Computes exact 3-player Shapley values ($2^3=8$ coalitions) measuring how much each modality drives predictions across all 14 diseases.

In [ ]:
from src.explainability.modality_attribution import MultimodalAttribution
from src.dataset.label_encoder import DISEASE_LABEL_NAMES

mm_attr = MultimodalAttribution(model, num_classes=14)
attr_res = mm_attr.attribute_instance(sample_images, sample_meta, gx, edge_index)

# Compute mean absolute contribution across cohort
mean_I = np.abs(attr_res['Image']).mean(axis=0)
mean_M = np.abs(attr_res['Clinical_Meta']).mean(axis=0)
mean_G = np.abs(attr_res['Graph']).mean(axis=0)
total = mean_I + mean_M + mean_G + 1e-8

pct_I = (mean_I / total) * 100
pct_M = (mean_M / total) * 100
pct_G = (mean_G / total) * 100

modality_df = pd.DataFrame({
    'Disease': DISEASE_LABEL_NAMES,
    'Image (%)': np.round(pct_I, 2),
    'Clinical_Meta (%)': np.round(pct_M, 2),
    'Graph (%)': np.round(pct_G, 2),
})
print('Multimodal Fusion Attribution Across Held-Out Test Cohort:')
print(modality_df.to_string(index=False))

fig = ExplainabilityVisualizer.plot_modality_contributions(
    pct_I, pct_M, pct_G, DISEASE_LABEL_NAMES, title='Multimodal Attribution by Modality across 14 Diseases'
)
plt.show()


## Cell 8 — Quantitative Faithfulness Evaluation
Measures explanation validity via pixel deletion/insertion curves, tabular perturbations, and graph edge removal.

In [ ]:
from src.explainability.faithfulness import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(model)
target_disease_idx = target_diseases[0][1]
target_disease_name = target_diseases[0][0]

# 1. Image Faithfulness (Deletion Curves on primary scan)
img_faith = evaluator.evaluate_image_faithfulness(
    sample_images[:1],
    hmap,
    target_class=target_disease_idx,
    meta=sample_meta[:1],
    graph_x=sample_meta[:1],
    graph_edge_index=torch.zeros((2, 1), dtype=torch.long, device=device),
    steps=[0.1, 0.2, 0.3, 0.4, 0.5]
)

# 2. Demographic Feature Faithfulness (Targeted Age/Sex Perturbation)
tab_faith = evaluator.evaluate_tabular_faithfulness(
    image=sample_images[:1],
    meta=sample_meta[:1],
    target_class=target_disease_idx,
    shap_values=shap_results
)

# 3. Graph Faithfulness (Edge Drop Impact on multi-scan graph)
graph_faith = evaluator.evaluate_graph_faithfulness(
    sample_images,
    sample_meta,
    gx,
    edge_index,
    np.array(graph_exp['edge_importance']),
    target_class=target_disease_idx
)

# 4. Modality Faithfulness (Dominant Modality Ablation)
modality_pcts = np.stack([pct_I, pct_M, pct_G], axis=0)  # [3, 14]
mod_faith = evaluator.evaluate_modality_faithfulness(
    sample_images,
    sample_meta,
    gx,
    edge_index,
    modality_pcts,
    target_class=target_disease_idx
)

print(f"=== Quantitative Faithfulness Metrics: '{target_disease_name}' ===")
print(f"Image Faithfulness:       {'✓ Confirmed (Salient pixel drop > Random)' if img_faith['is_faithful'] else 'Partial Sensitivity'}")
print(f" - Max Salient Drop:      {max(img_faith['most_important_drops']):.4f} vs Random: {max(img_faith['random_drops']):.4f}")
print(f"Tabular Faithfulness:     {'✓ Confirmed (Top feature drop >= Other)' if tab_faith['is_faithful'] else 'Partial'}")
print(f" - Top Feature ({tab_faith['top_feature']}): {tab_faith['top_feature_delta']:.4f} vs Other: {tab_faith['other_feature_delta']:.4f}")
print(f"Graph Faithfulness:       {'✓ Confirmed (Important edge drop >= Random)' if graph_faith['is_faithful'] else 'Partial'}")
print(f" - Important Edge Drop:   {graph_faith['important_edges_drop']:.4f} vs Random: {graph_faith['random_edges_drop']:.4f}")
print(f"Modality Faithfulness:    {'✓ Confirmed (Dominant ablation largest)' if mod_faith['is_faithful'] else 'Balanced'}")
print(f" - Dominant Modality:     {mod_faith['dominant_attributed_modality']}")
print(f" - Modality Drops:        Image={mod_faith['modality_drops']['Image']:.4f}, Meta={mod_faith['modality_drops']['Clinical_Meta']:.4f}, Graph={mod_faith['modality_drops']['Graph']:.4f}")

# Plot Image Deletion Curves
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(img_faith['steps'], img_faith['most_important_drops'], 'r-o', label='Most Important (Salient) Pixels')
ax.plot(img_faith['steps'], img_faith['random_drops'], 'b--^', label='Random Pixels (Control)')
ax.plot(img_faith['steps'], img_faith['least_important_drops'], 'g:s', label='Least Important Pixels')
ax.set_xlabel('Fraction of Pixels Masked', fontsize=11)
ax.set_ylabel('Probability Drop (|Base - Perturbed|)', fontsize=11)
ax.set_title(f'Image Faithfulness Deletion Curves ({target_disease_name})', fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
